# Figure 4: Representations at convergence

In [1]:
import numpy as np
import seaborn as sns
from networks.linear_network import LinearNetwork
from utils import get_random_regression_task, get_lambda_balanced, get_ntk, kernel_distance,BlindColours
from qqt.lambda_balanced_extra_dim_clean import QQT_lambda_balanced_unequal
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from qqt.lambda_balanced_extra_dim_clean import MTL_QQT_lambda_balanced_bottleneck
from qqt.lambda_balanced_extra_dim_bottel import QQT_lambda_balanced_unequal
from utils import get_random_regression_task, get_lambda_balanced,get_lambda_balanced_botteld, get_ntk, reshape_matrix, kernel_distance, cosine_similarity,BlindColours, zero_balanced_weights
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
from string import ascii_lowercase as _abc
import torch
import torch.nn.functional as F
from matplotlib import ticker
from matplotlib.ticker import FormatStrFormatter

(array([[ 1.165874  ,  0.22676856, -0.10736271],
       [-0.57332103,  1.21811682,  0.5187243 ],
       [ 0.40023123, -0.77389113,  0.97228567]]), array([[ 0.24425528,  0.34006998, -0.62750218],
       [-0.40046948,  0.22921536, -0.42338566],
       [ 0.45767659, -0.95952312,  0.3724365 ]]), array([[2.20227832, 0.        , 0.        ],
       [0.        , 0.68862557, 0.        ],
       [0.        , 0.        , 0.35934338]]), array([[ 9.92708598e-01, -1.11022302e-16, -8.32667268e-17],
       [-1.11022302e-16,  9.92708598e-01,  4.44089210e-16],
       [-8.32667268e-17,  4.44089210e-16,  9.92708598e-01]]))
Sisdezegoed [4.66555906e-08 2.20600488e-08]
Shi [[1.00000002e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 8.57516747e-05 0.00000000e+00 0.00000000e+00]]
S_1 [[0.000216 0.      ]
 [0.       0.      ]]
U [[-0.88592346 -0.46383145]
 [-0.46383145  0.88592346]]
sxi
w1 [[1.88804071e-04 3.11632109e-05]
 [9.88496977e-05 1.63157179e-05]]
w2 [[ 0.1308323  -0.79265488]
 [-0.

## MTL DATA GENERATION

In [ ]:

from utils import whiten
from utils import MTL_get_lambda_balanced_STLex
from qqt.lambda_balanced_extra_dim_clean import MTL_QQT_lambda_balanced_unequal
import random


seed=5434
random.seed(seed)
np.random.seed(seed)
np.random.default_rng(seed=seed)

learning_rate = 0.00002
training_steps = 20000
in_dim=4
out_dim_1=1
out_dim_2=1
hidden_dim=2
batch_size=100



def get_random_regression_task_MTL(batch_size, in_dim, out_dim_1, out_dim_2):
    X = np.random.randn(batch_size, in_dim)
    Y1 = np.random.randn(batch_size, out_dim_1)
    Y2= np.random.randn(batch_size, out_dim_2)
    X_whitened = whiten(X)
    return X_whitened, Y1, Y2


def get_MTL_Tasks(batch_size, in_dim, out_dim_1, out_dim_2, alpha_scales=(1,1), alignment_factor=0.3, whitenli=True, seed=31):
    rng = np.random.default_rng(seed)

    X = np.random.randn(batch_size, in_dim)

    Q_full, _ = np.linalg.qr(rng.standard_normal((in_dim, in_dim)))
    W1=Q_full[:, :out_dim_1].T
    W2 = alignment_factor*W1 + (1-alignment_factor)*Q_full[:, out_dim_1:2*out_dim_1].T
    W2 = W2 / np.linalg.norm(W2, 'fro') * np.linalg.norm(W1, 'fro')
    W1= W1/ np.linalg.norm(W1, 'fro')*np.linalg.norm(W2,'fro')
    if whitenli == True:
        X = whiten(X)
    Y1 = alpha_scales[0]*W1 @ X.T
    Y2 = alpha_scales[1]*W2 @ X.T

    if out_dim_1 == 1:
      Y1=Y1.reshape(1, batch_size)
      Y2=Y2.reshape(1, batch_size)

    return X, Y1, Y2


def get_MTL_Tasks_spectrum(batch_size, in_dim, alphas, alpha_scales=(1,1),
                           whitenli=True, seed=None):
    """Two linear tasks whose overlap matrix O_12 has singular values exactly `alphas`.
       alphas[k] in [0,1]: 1 = shared mode, 0 = orthogonal, in-between = conflicting.
       Requires 2*len(alphas) <= in_dim  (i.e. N_o1 + N_o2 < N_1)."""
    rng = np.random.default_rng(seed)
    alphas = np.asarray(alphas, float)
    d = len(alphas)
    assert 2*d <= in_dim, f"need 2*{d} <= {in_dim} (N_o1+N_o2 < N_1)"
    assert np.all((alphas >= 0) & (alphas <= 1)), "alphas must be in [0,1]"

    X = np.random.randn(batch_size, in_dim)
    Q, _ = np.linalg.qr(rng.standard_normal((in_dim, in_dim)))
    A, B = Q[:, :d], Q[:, d:2*d]                       # mutually orthonormal blocks
    W1 = A.T
    W2 = alphas[:, None]*A.T + np.sqrt(1-alphas**2)[:, None]*B.T

    if whitenli:
        X = whiten(X)
    Y1 = alpha_scales[0]*W1 @ X.T
    Y2 = alpha_scales[1]*W2 @ X.T
    return X, Y1, Y2
    

## Expected Representations for different alignments

In [3]:
ali_us_1=[]
ali_us_1_std=[]
ali_other=[]
ali_other_std=[]
magdif=[]
magdif_std = []
S_1=[]
S_2=[]
Sxi=[]
V_1=[]
V_2=[]
V_xilist=[]
U_1=[]
U_2=[]
U_xilist=[]
USU_1=[]
USU_2=[]
USU_MTL=[]
VSV_1=[]
VSV_2=[]
VSV_MTL=[]
USV_1=[]
USV_2=[]
USV_MTL=[]



#for alpha2 in [1,1.2, 1.3,1.5,2,5,10]:
#for Sigma  in  [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6,0.7,0.8, 0.9, 1, 1.5, 2, 2.5, 3,3.5, 4,4.5, 5,5.5,6,7,8,9,10,12,14,16,18,20] :
alpha2=1
ali_range=[1, 0.5, 0]
signu=[]
signv=[]
for ali in ali_range:
 ali_us_seed=[]
 ali_other_seed=[]
 magdif_seed=[]
 U1_seed=[]
 U2_seed=[]
 Uxi_seed=[]
 V1_seed=[]
 V2_seed=[]
 Vxi_seed=[]
 Sxi_seed=[]
 signu_seed=[]
 signv_seed=[]
 VSV_1_seed=[]
 USU_1_seed=[]
 VSV_2_seed=[]
 USU_2_seed=[]
 VSV_MTL_seed=[]
 USU_MTL_seed=[]
 Soverlap=[]
 Soverlap_seed=[]
 USV_1_seed=[]
 USV_2_seed=[]
 USV_MTL_seed=[]


 USU_1_std = []
 USU_2_std = []
 USU_MTL_std = []

 VSV_1_std = []
 VSV_2_std = []
 VSV_MTL_std = []
 VSV_delta_std = []
 for seed in  [random.randint(1, 1000) for _ in range(200)]:
    random.seed(seed)
    np.random.seed(seed)
    D, N1, N2 = 4, 1,1
    rho = 1
    α1, α2 = 1.0, 1
    S = 5000
    noise = 0
    sigma_eps=5
    
    X, Y1, Y2= get_MTL_Tasks( S, D, N1, N2, alpha_scales=(1,alpha2), alignment_factor=ali, seed=seed)

    print(Y1.shape, Y2.shape)
    Sigma1o = Y1 @ X/ S # (output_dim, input_dim)
    Sigma2o = Y2 @ X / S

    U1, S1, V1=np.linalg.svd(Sigma1o, full_matrices=False)
    S_1.append(S1)
    U1_seed.append(U1)
    V1_seed.append(V1)

    U2, S2, V2=np.linalg.svd(Sigma2o, full_matrices=False)
    S_2.append(S2)
    U2_seed.append(U2)
    V2_seed.append(V2)

    U, S, V=np.linalg.svd(V1@V2.T)
    print("alignment valuee", S)
    signv_seed.append(V1@V2.T)
    ali_us_seed.append(S)
    Uextra, Sextra, Vextra=np.linalg.svd(np.dot(U1,U2))
    signu_seed.append(U1@U2)

    XI=np.concatenate((Sigma1o, Sigma2o), axis=0)
    U_xi,S_xi,Vt_xi=np.linalg.svd(XI,full_matrices=False)
    Uxi_seed.append(U_xi)
    Vxi_seed.append(Vt_xi)
    Sxi_seed.append(S_xi)
    m1 = U1.shape[0]
    m2 = U2.shape[0]
    r  = S_xi.size
    Sxi.append(S_xi)
    # build original energy‐matrices
    M1 = U1  @ np.diag(S1)  @ U1.T
    USU_1_seed.append(M1)

    M2 = U2  @ np.diag(S2)  @ U2.T
    USU_2_seed.append(M2)

    Mxi = U_xi @ np.diag(S_xi) @ U_xi.T
    USU_MTL_seed.append(Mxi)

    Mxi1= (U_xi@ np.diag(S_xi)@U_xi.T)[:m1, :m1]
    Mxi2=(U_xi@ np.diag(S_xi)@U_xi.T)[m1:m1+m2, m1:m1+m2]

    # split pooled U_xi
    U1_xi = U_xi[:m1, :m1]   # (m1 x r)
    U2_xi = U_xi[m1:, m1:]   # (m2 x r)


    USV_2_seed.append(U2@np.diag(S2)@V2)
    USV_1_seed.append(U1@np.diag(S1)@V1)
    USV_MTL_seed.append(U_xi@np.diag(S_xi)@Vt_xi)

    # Third row: VSV^T
    V1SV1 = V1.T @ np.diag(S1) @ V1
    VSV_1_seed.append(V1SV1)
    V2SV2 = V2.T @ np.diag(S2) @ V2
    VSV_2_seed.append(V2SV2)
    VxiSVxi = Vt_xi.T @ np.diag(S_xi) @ Vt_xi
    VSV_MTL_seed.append(VxiSVxi)
    vmax = max(M1.max(), M2.max(), Mxi1.max(),Mxi2.max())

    print("othernotion1", np.dot(Sigma2o[0,:],Sigma1o[0,:])/(np.linalg.norm(Sigma1o[0,:])*np.linalg.norm(Sigma2o[0,:])))
    ali_other_seed.append(np.dot(Sigma2o[0,:],Sigma1o[0,:])/(np.linalg.norm(Sigma1o[0,:])*np.linalg.norm(Sigma2o[0,:])))

    print("magdif1_12", np.linalg.norm(Sigma1o[0,:])/np.linalg.norm(Sigma2o[0,:]))
    magdif_seed.append(np.linalg.norm(Sigma1o[0,:])/np.linalg.norm(Sigma2o[0,:]))

 ali_us_1.append(np.mean(ali_us_seed))
 ali_us_1_std.append(np.std(ali_us_seed))
 ali_other.append(np.mean(ali_other_seed))
 ali_other_std.append(np.std(ali_other_seed))
 magdif.append(np.mean(magdif_seed))
 magdif_std.append(np.std(magdif_seed))
 U_1.append(np.mean(U1_seed, axis=0))
 U_2.append(np.mean(U2_seed, axis=0))
 U_xilist.append(np.mean(Uxi_seed, axis=0))
 V_1.append(np.mean(V1_seed, axis=0))
 V_2.append(np.mean(V2_seed, axis=0))
 V_xilist.append(np.mean(Vxi_seed, axis=0))
 signu.append(np.mean(signu_seed, axis=0))
 signv.append(np.mean(signv_seed, axis=0))
 Sxi.append(np.mean(Sxi, axis=0))
 USU_1.append(np.mean(USU_1_seed, axis=0))
 USU_2.append(np.mean(USU_2_seed, axis=0))
 USU_MTL.append(np.mean(USU_MTL_seed, axis=0))
 VSV_1.append(np.mean(VSV_1_seed, axis=0))
 VSV_2.append(np.mean(VSV_2_seed, axis=0))
 VSV_MTL.append(np.mean(VSV_MTL_seed, axis=0))
 USV_1.append(np.mean(USV_1_seed, axis=0))
 USV_2.append(np.mean(USV_2_seed, axis=0))
 USV_MTL.append(np.mean(USV_MTL_seed, axis=0))
 USU_1_std.append(np.std(USU_1_seed, axis=0))
 USU_2_std.append(np.std(USU_2_seed, axis=0))
 USU_MTL_std.append(np.std(USU_MTL_seed, axis=0))
 
 VSV_1_std.append(np.std(VSV_1_seed, axis=0))
 VSV_2_std.append(np.std(VSV_2_seed, axis=0))
 VSV_MTL_std.append(np.std(VSV_MTL_seed, axis=0))
 VSV_delta_std.append(
    np.std(
        np.asarray(VSV_MTL_seed) - np.asarray(VSV_1_seed) - np.asarray(VSV_2_seed),
        axis=0
    )
)

(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 1.0
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 1.0
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 1.0
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 0.9999999999999999
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 1.0
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 0.9999999999999999
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 1.0
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 0.9999999999999999
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 0.9999999999999999
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 0.9999999999999999
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 1.0
magdif1_12 1.0
(1, 5000) (1, 5000)
alignment valuee [1.]
othernotion1 0.9999999999999999
magdif1_12 1.0
(1, 5000) (1, 5000)
al

### USU

In [ ]:
# ----- print-friendly style -----
mpl.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 12, "axes.titlesize": 12, "axes.labelsize": 8,
    "axes.linewidth": 0.5
})

# ----- color scales -----
corr_vmax  = 1.3
norm_corr  = mpl.colors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=corr_vmax)
cmap       = mpl.cm.get_cmap("viridis")

# Δ scale symmetric to max |Δ|
def _max_abs_delta(USU_1, USU_2, USU_MTL):
    vals = []
    for i in range(len(USU_1)):
        base = np.block([[USU_1[i],                np.zeros_like(USU_1[i])],
                         [np.zeros_like(USU_2[i]), USU_2[i]]])
        vals.append(np.abs(np.diag(USU_MTL[i]-base) / np.diag(base)))
    return float(np.max(vals))
dmax        = _max_abs_delta(USU_1, USU_2, USU_MTL)
norm_delta  = mpl.colors.TwoSlopeNorm(vmin=-0.7, vcenter=0, vmax=0.7)

# ----- helpers -----
def _lum(rgb):
    r,g,b,_ = rgb
    to_lin = lambda u: u/12.92 if u <= 0.04045 else ((u+0.055)/1.055)**2.4
    R,G,B = map(to_lin, (r,g,b))
    return 0.2126*R + 0.7152*G + 0.0722*B

def add_panel_label(ax, label, x=0.75, y=0.98):
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="top",
            fontsize=9, fontweight="bold")

# ----- figure: left sidebar + 2-row main grid -----
n_g = len(ali_range)
fig = plt.figure(figsize=(n_g + 2, 2.7), constrained_layout=True)
gs  = gridspec.GridSpec(nrows=2, ncols=3,
                        width_ratios=[1.1, n_g, 0.06],
                        height_ratios=[1, 1],
                        figure=fig)

letters = iter(_abc)  # (a), (b), ...

# Sidebar: USU-1 (top), USU-2 (bottom)

ax_ref1 = fig.add_subplot(gs[0, 0])
ax_ref2 = fig.add_subplot(gs[1, 0])
target_shape = USU_MTL[0].shape  # shape used by the main panels

def to_2d(A):
    A = np.asarray(A)
    if A.ndim == 0:
        return A.reshape(1, 1)
    if A.ndim == 1:
        return A.reshape(1, -1)
    return A

def tile_to_shape(A, shape):
    A = to_2d(A)
    reps = (int(np.ceil(shape[0] / A.shape[0])),
            int(np.ceil(shape[1] / A.shape[1])))
    out = np.tile(A, reps)
    return out[:shape[0], :shape[1]]

ax_ref1.imshow(tile_to_shape(USU_1[0], target_shape),
               cmap=cmap, norm=norm_corr, interpolation="nearest", rasterized=True)
ax_ref2.imshow(tile_to_shape(USU_2[0], target_shape),
               cmap=cmap, norm=norm_corr, interpolation="nearest", rasterized=True)

for a, data, title in [(ax_ref1, tile_to_shape(USU_1[0], target_shape), "$U_1S_1U_1^T$"), (ax_ref2, tile_to_shape(USU_2[0], target_shape), "$U_2S_2U_2^T$")]:
    a.imshow(data, cmap=cmap, norm=norm_corr, interpolation="nearest", rasterized=True)
    #annotate_diagonal(a, data, norm=norm_corr, cmap=cmap)
    #a.set_title(title, pad=3)
    a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal")
    for s in a.spines.values(): s.set_visible(False)
    add_panel_label(a, f"({next(letters)})")

# Main grid: 2 rows × n_g columns
sub = gs[:, 1].subgridspec(2, n_g, wspace=0.04, hspace=0.10)
ax_mtl   = [fig.add_subplot(sub[0, j]) for j in range(n_g)]
ax_delta = [fig.add_subplot(sub[1, j]) for j in range(n_g)]

for i, ali in enumerate(ali_range):
    baseline = np.block([[USU_1[i],                np.zeros_like(USU_1[i])],
                         [np.zeros_like(USU_2[i]), USU_2[i]]])
    #rel = np.diag(USU_MTL[i] - baseline) / np.diag(baseline)
    rel=USU_MTL[i]-baseline
    # Top row: USU-MTL (title on EACH column)
    ax_mtl[i].imshow(USU_MTL[i], cmap=cmap, norm=norm_corr, interpolation="nearest", rasterized=True)
    #annotate_diagonal(ax_mtl[i], USU_MTL[i], norm=norm_corr, cmap=cmap)
    #ax_mtl[i].set_title("$U_\\xi S_\\xi U^T_\\xi$", pad=3)

    # Bottom row: ΔUSU (title on EACH column)
    D = rel
    ax_delta[i].imshow(D, cmap=cmap, norm=norm_delta, interpolation="nearest", rasterized=True)
    #annotate_diagonal(ax_delta[i], D, norm=norm_delta, cmap=cmap)
    #ax_delta[i].set_title(r"$\Delta$ $USU^T$", pad=3)

    # Cosmetics + γ labels under columns
    for a in (ax_mtl[i], ax_delta[i]):
      a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal")
     #   for s in a.spines.values(): s.set_visible(False)
    #ax_delta[i].set_xlabel(r"$\gamma$="+str(ali), labelpad=2)

    # Panel letters (continue sequence)
    add_panel_label(ax_mtl[i],   f"({next(letters)})")
    add_panel_label(ax_delta[i], f"({next(letters)})")

# Colorbars on the right (one per row)
cax1 = fig.add_subplot(gs[0, 2])
cax2 = fig.add_subplot(gs[1, 2])
cb1 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm_corr,  cmap=cmap), cax=cax1)
cb2 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm_delta, cmap=cmap), cax=cax2)


#cb1.set_label("Correlation Strength", fontsize=8)
#cb2.set_label(r"Relative Change", fontsize=8)
for cb in (cb1, cb2):
    cb.ax.tick_params(length=2.5, width=0.5, pad=2)
    cb.outline.set_linewidth(0.5)

# fig.savefig("usu_two_row_labeled.pdf", bbox_inches="tight")
plt.show()


### VSV

In [ ]:

# --- helpers ---
def annotate_diagonal(ax, M, fmt="{:.2f}", *, norm=None, cmap=None, fs=7.5):
    # auto-contrast text on any background
    def _lum(rgb):
        r,g,b,_ = rgb
        to_lin = lambda u: u/12.92 if u<=0.04045 else ((u+0.055)/1.055)**2.4
        R,G,B = map(to_lin,(r,g,b))
        return 0.2126*R+0.7152*G+0.0722*B
    for k, v in enumerate(np.diag(M)):
        bg = cmap(norm(v)) if (norm and cmap) else (1,1,1,1)
        color = "black" if _lum(bg) > 0.45 else "white"
        halo  = "white" if color == "black" else "black"
        ax.text(k, k, fmt.format(v), ha="center", va="center",
                fontsize=fs, fontweight="semibold", color=color,
                path_effects=[pe.Stroke(linewidth=1, foreground=halo), pe.Normal()],
                clip_on=True)

def hide_axes_but_keep_labels(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_aspect('equal')
def add_panel_label(ax, label, x=0.78, y=0.98):
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="top",
            fontsize=9, fontweight="bold")

# --- build data-dependent norms (better contrast) ---
n_g = len(ali_range)

# correlation (VSV×SI) scale: center at 0, use data min/max
corr_vals_min = min(np.min(VSV_MTL[j]) for j in range(n_g))
corr_vals_max = max(np.max(VSV_MTL[j]) for j in range(n_g))
norm_corr = mpl.colors.TwoSlopeNorm(vmin=float(corr_vals_min), vcenter=0, vmax=float(corr_vals_max))

# delta scale: symmetric about 0 using max |Δ|
dmax = max(np.max(np.abs(VSV_MTL[j] - VSV_1[j] - VSV_2[j])) for j in range(n_g))
norm_delta = mpl.colors.TwoSlopeNorm(vmin=-float(dmax), vcenter=0, vmax=float(dmax))

# diverging cmap reads differences better than 'viridis'
cmap = mpl.cm.get_cmap("viridis")

fig_h = 2.7      # tweak as you like
fig_w = 1.2                       # narrow column
fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=False)

gs = gridspec.GridSpec(
    nrows=n_g, ncols=2,           # last col = colorbar
    width_ratios=[1, 0.10],
    height_ratios=[1]*n_g,
    wspace=0.06, hspace=0.08, figure=fig
)

# axes for panels
# one axis per row (column layout)
axes = [fig.add_subplot(gs[i, 0]) for i in range(n_g)]

# --- draw ONLY the bottom-row content (Δ VSV) ---
ims = []
for j, ax in enumerate(axes):
    rel = VSV_MTL[j] - VSV_1[j] - VSV_2[j]
    im = ax.imshow(rel, cmap=cmap, norm=norm_delta, interpolation='nearest', rasterized=True)
    ims.append(im)
    hide_axes_but_keep_labels(ax)
    add_panel_label(ax, f"({_abc[j]})")  # panel letters (A, B, C, ...)
# row labels (only once, left-most panels)

for cb in (cb1, cb2):
    cb.outline.set_linewidth(0.5)
    cb.ax.tick_params(length=2.5, width=0.5, pad=2)
#fig.text(0.02, 0.5, r"$\Delta$ VSV", va="center", rotation=90, fontsize=9)

# --- a single colorbar spanning all rows ---
# --- a single compact colorbar spanning all rows ---
cax = fig.add_subplot(gs[:, 1])  # span all rows
cb  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_delta, cmap=cmap),
    cax=cax
)

# make it slimmer
cax.set_position([
    cax.get_position().x0,
    cax.get_position().y0,
    cax.get_position().width * 0.5,  # shrink width
    cax.get_position().height
])

# only three ticks: min, 0, max
cb.set_ticks([norm_delta.vmin, 0, norm_delta.vmax])
cb.ax.set_yticklabels(["-0.5", "0", "0.5"])
# instead of fig.text(... rotation=90)

fig.text(0.475, 0.94, r"$\Delta$ $VSV^T$",
         ha="center", va="top", fontsize=9)
# style
cb.outline.set_linewidth(0.5)
cb.ax.tick_params(length=2.0, width=0.5, pad=1.5, labelsize=8)


## Appendix; Mixed spectra

In [ ]:
ali_us_1 = []
ali_us_1_std = []
ali_other = []
ali_other_std = []
magdif = []
magdif_std = []

S_1 = []
S_2 = []
Sxi = []

V_1 = []
V_2 = []
V_xilist = []

U_1 = []
U_2 = []
U_xilist = []

USU_1 = []
USU_2 = []
USU_MTL = []

VSV_1 = []
VSV_2 = []
VSV_MTL = []

USV_1 = []
USV_2 = []
USV_MTL = []

signu = []
signv = []

# Parameters
D, N1, N2 = 10, 4, 4
num_samples = 10
alpha1, alpha2 = 1.0, 1.0
noise = 0
sigma_eps = 5
rho = 1

ali_range = [1, 0.5, 0]
n_seeds = 200

# Example mixed spectra.
# Adjust these to the spectra you want get_MTL_Tasks_spectrum to use.
spectrum_1 = np.array([1.0, 0.8, 0.75, 0.68])
spectrum_2 = np.array([1, 0.2,0.14,0.12])


ali_range=[spectrum_1, spectrum_2]
for ali in ali_range:
    ali_us_seed = []
    ali_other_seed = []
    magdif_seed = []

    U1_seed = []
    U2_seed = []
    Uxi_seed = []

    V1_seed = []
    V2_seed = []
    Vxi_seed = []

    S1_seed = []
    S2_seed = []
    Sxi_seed = []

    signu_seed = []
    signv_seed = []

    USU_1_seed = []
    USU_2_seed = []
    USU_MTL_seed = []

    VSV_1_seed = []
    VSV_2_seed = []
    VSV_MTL_seed = []

    USV_1_seed = []
    USV_2_seed = []
    USV_MTL_seed = []

    for seed in [random.randint(1, 100) for _ in range(n_seeds)]:
        np.random.seed(seed)
        random.seed(seed)

        X, Y1, Y2 = get_MTL_Tasks_spectrum(
        num_samples,
            D,  
        ali,
        alpha_scales=(alpha1, alpha2),
        whitenli=True,
         seed=seed,
)

        print("Y shapes:", Y1.shape, Y2.shape)

        Sigma1o = Y1 @ X / num_samples
        Sigma2o = Y2 @ X / num_samples

        U1, S1, Vt1 = np.linalg.svd(Sigma1o, full_matrices=False)
        U2, S2, Vt2 = np.linalg.svd(Sigma2o, full_matrices=False)

        S1_seed.append(S1)
        S2_seed.append(S2)

        U1_seed.append(U1)
        U2_seed.append(U2)

        V1_seed.append(Vt1)
        V2_seed.append(Vt2)

        _, Sv_align, _ = np.linalg.svd(Vt1 @ Vt2.T)
        print("V alignment:", Sv_align)

        signv_seed.append(Vt1 @ Vt2.T)
        ali_us_seed.append(Sv_align)

        signu_seed.append(U1.T @ U2)

        XI = np.concatenate((Sigma1o, Sigma2o), axis=0)
        U_xi, S_xi, Vt_xi = np.linalg.svd(XI, full_matrices=False)

        Uxi_seed.append(U_xi)
        Vxi_seed.append(Vt_xi)
        Sxi_seed.append(S_xi)

        M1 = U1 @ np.diag(S1) @ U1.T
        M2 = U2 @ np.diag(S2) @ U2.T
        Mxi = U_xi @ np.diag(S_xi) @ U_xi.T

        USU_1_seed.append(M1)
        USU_2_seed.append(M2)
        USU_MTL_seed.append(Mxi)

        USV_1_seed.append(U1 @ np.diag(S1) @ Vt1)
        USV_2_seed.append(U2 @ np.diag(S2) @ Vt2)
        USV_MTL_seed.append(U_xi @ np.diag(S_xi) @ Vt_xi)

        V1SV1 = Vt1.T @ np.diag(S1) @ Vt1
        V2SV2 = Vt2.T @ np.diag(S2) @ Vt2
        VxiSVxi = Vt_xi.T @ np.diag(S_xi) @ Vt_xi

        VSV_1_seed.append(V1SV1)
        VSV_2_seed.append(V2SV2)
        VSV_MTL_seed.append(VxiSVxi)

        sigma1_vec = Sigma1o.reshape(-1)
        sigma2_vec = Sigma2o.reshape(-1)

        cosine_sigma = np.dot(sigma1_vec, sigma2_vec) / (
            np.linalg.norm(sigma1_vec) * np.linalg.norm(sigma2_vec)
        )
        mag_ratio = np.linalg.norm(sigma1_vec) / np.linalg.norm(sigma2_vec)

        print("Sigma cosine:", cosine_sigma)
        print("Magnitude ratio:", mag_ratio)

        ali_other_seed.append(cosine_sigma)
        magdif_seed.append(mag_ratio)

    ali_us_1.append(np.mean(ali_us_seed, axis=0))
    ali_us_1_std.append(np.std(ali_us_seed, axis=0))

    ali_other.append(np.mean(ali_other_seed))
    ali_other_std.append(np.std(ali_other_seed))

    magdif.append(np.mean(magdif_seed))
    magdif_std.append(np.std(magdif_seed))

    S_1.append(np.mean(S1_seed, axis=0))
    S_2.append(np.mean(S2_seed, axis=0))
    Sxi.append(np.mean(Sxi_seed, axis=0))

    U_1.append(np.mean(U1_seed, axis=0))
    U_2.append(np.mean(U2_seed, axis=0))
    U_xilist.append(np.mean(Uxi_seed, axis=0))

    V_1.append(np.mean(V1_seed, axis=0))
    V_2.append(np.mean(V2_seed, axis=0))
    V_xilist.append(np.mean(Vxi_seed, axis=0))

    signu.append(np.mean(signu_seed, axis=0))
    signv.append(np.mean(signv_seed, axis=0))

    USU_1.append(np.mean(USU_1_seed, axis=0))
    USU_2.append(np.mean(USU_2_seed, axis=0))
    USU_MTL.append(np.mean(USU_MTL_seed, axis=0))

    VSV_1.append(np.mean(VSV_1_seed, axis=0))
    VSV_2.append(np.mean(VSV_2_seed, axis=0))
    VSV_MTL.append(np.mean(VSV_MTL_seed, axis=0))

    USV_1.append(np.mean(USV_1_seed, axis=0))
    USV_2.append(np.mean(USV_2_seed, axis=0))
    USV_MTL.append(np.mean(USV_MTL_seed, axis=0))